# MongoDB Handling

After installing the MongoDB server in your machine, you can use this notebook for handling the initial processes with the database.

Specifically, in this step, we utilize Python's `pymongo` library to exploit its capabilities for MongoDB server interaction.

**Important Note: Be sure that the MongoDB server is up and running as a service in the background.**

For example, in macOS, to run MongoDB (i.e. the mongod process) as a service, run:

* `brew services start mongodb-community`

To stop a mongod running as a macOS service, use the following command as needed:

* `brew services stop mongodb-community`

To install MongoDB in your system, follow the instructions here:

* https://www.mongodb.com/docs/manual/administration/install-community/


**Note:** You can modify any of the processes below, however, you have to explain your thoughts.

In [15]:
# import library for various processes with the OS
import os

## Load configuration

In [16]:
# import library for yaml handling
import yaml

In [17]:
config_path = os.path.join(os.getcwd(), "config.yml")

with open(config_path) as file:
    config = yaml.load(file, Loader=yaml.FullLoader)

## MongoDB database instantiation

The relevant information for the MongoDB client connection, the database name, and collection name is located in the configuration file.

```
# DB Connection with the uri (host)
client: "mongodb://localhost:27017/"

# db name
db: "aiot_course"

# db collection
col: "NAME YOUR COLLECTION"
```

In [18]:
# import library for hanlding the MongoDB client
import pymongo
# import library for retrieving datetime
from datetime import datetime

### Create the database

To create a database in MongoDB, start by creating a MongoClient object, then specify a connection URL with the correct ip address and the name of the database you want to create.

MongoDB will create the database if it does not exist, and make a connection to it.

In [19]:
client = pymongo.MongoClient(config["client"])

In [20]:
db = client[config["db"]]

### Instantiate the collection

To create a collection in MongoDB, use the database object and specify the name of the collection you want to create.

MongoDB will create the collection if it does not exist.

In [21]:
col = db[config["col"]]

Initially, no collection will be shown in MongoDB before you enter the first document!

## Create the data collection

Uploading the gathered data to MongoDB collection. The data directory structure should be as follows:

```
.
└── data/
    ├──
    ├── scroll-up-thumb/
    │   ├── scroll-up-thumb_0_50_AccGyr_1_1_01_61964f51d11ec26c3bbde60b.csv
    │   ├── scroll-up-thumb_0_50_AccGyr_1_1_01_7982f51d11ec26c3be60b875.csv
    │   └── ..
    ├── scroll-up-down/
    │   ├── scroll-up-down_0_50_AccGyr_1_1_01_12432f51d11ec26c3b944jd0.csv
    │   ├── scroll-up-down_0_50_AccGyr_1_1_02_47563f51d11ec26c3bb4210k.csv
    │   └── .
    └── class ...
```

In [22]:
# import library for handling the csv data and transformations
import pandas as pd
import json

Get data path:

In [23]:
data_path = os.path.join(os.getcwd(), "data")
print(data_path)

/Users/alexgiann/GitHub/CEID/8th Semester/IOT/social-media-gesture-recognition/data


List all files in a path:

In [24]:
classes_folders_list = [f for f in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, f))]
print(classes_folders_list)

['texting', 'scroll-down', 'scroll-up', 'swipe-right', 'swipe-left']


In [25]:
# print files in folder
folder_path = os.path.join(data_path, classes_folders_list[0])
files_in_folder = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
print(files_in_folder)

['texting_two_0_100_gyr_0_b.csv', 'texting_two_0_100_acc_0_b.csv', 'texting_two_0_100_gyr_0_a.csv', 'texting_two_0_100_acc_0_a.csv', 'texting_two_0_100_gyr_0_s.csv', 'texting_two_0_100_acc_0_s.csv']


Each document in the MongoDB database should have the following schema:

```json
{
  "_id": ObjectId("6984b3fa87abe7f4dff571aa"),
  "data": {
    "acc_x": ["array", "of", "values"],
    "acc_y": ["array", "of", "values"],
    "acc_z": ["array", "of", "values"]
  },
  "gesture_id": "The label of the instance",
  "hand": 0,
  "sr": 50,
  "sensor": "AccGyr",
  "primary": 1,
  "spontaneous": 1,
  "user": "01",
  "datetime": "MongoDB datetime object (it can be generated with the datetime.datetime.now() function"
}
```

Accordingly, if you are using gyroscope or both accelerometer and gyroscope, the following order and naming of the sensor keys should be defined:

* for gyroscope: `gyr_x`, `gyr_y`, `gyr_z` for the three axes
* for accelerometer and gyroscope: `acc_x`, `acc_y`, `acc_z`, `gyr_x`, `gyr_y`, `gyr_z` for the six axes

**Note: Be careful, the document is mandatory to have the aforementioned schema, in order to argue and proceed with the rest of the processes later on, in data engineering, plotting, etc.**

In [26]:
from utils import df_rebase

## Provide the code to upload the data to MongoDB

In [27]:
import os
import pandas as pd
from datetime import datetime

# 1. Define data path
data_path = os.path.join(os.getcwd(), "data")

documents_to_insert = []

# 2. Recursively search through all subfolders
for root, dirs, files in os.walk(data_path):
    for file_name in files:
        # 3. Process CSV files
        if file_name.endswith('.csv'):
            file_path = os.path.join(root, file_name)
            
            # Derive the gesture_id from the folder structure
            # Example: If root is '.../data/scroll-down/index' rel_path becomes 'scroll-down/index'
            rel_path = os.path.relpath(root, data_path)
            gesture_id = rel_path.replace(os.sep, '-')
            
            df = pd.read_csv(file_path)
            
            # Build the 'data' dictionary
            sensor_data = {}
            valid_columns = ['X', 'Y', 'Z']
            
            for column in df.columns:
                if column in valid_columns:
                    sensor_data[column] = df[column].tolist()
            
            # 4. Construct the MongoDB Document matching the schema
            if '_acc_' in file_name.lower() and '_gyr_' in file_name.lower():
                sensor = "AccGyr"
            elif '_acc_' in file_name.lower():
                sensor = "Acc"
            elif '_gyr_' in file_name.lower():
                sensor = "Gyr"
            else:
                sensor = "Unknown"

            user_map = {"a": "a", "b": "b", "s": "s"}
            user_key = os.path.splitext(file_name)[0].split("_")[-1]
            user = user_map.get(user_key, "Unknown")

            document = {
                "data": sensor_data,
                "gesture_id": gesture_id,
                "hand": 0,
                "sr": 100,
                "sensor": sensor,
                "primary": 1,
                "spontaneous": 0,
                "user": user,
                "datetime": datetime.now()
            }
            
            documents_to_insert.append(document)

# 5. Execute the bulk insert to MongoDB
if documents_to_insert:
    col.insert_many(documents_to_insert)
    print(f"✅ Success! Uploaded {len(documents_to_insert)} instances to the '{col.name}' collection.")
else:
    print("⚠️ No CSV files found. Check your folder structure.")

✅ Success! Uploaded 54 instances to the 'Sensor_Data' collection.


# Drop all collections in the current MongoDB database
for collection_name in db.list_collection_names():
    db.drop_collection(collection_name)
    print(f"Dropped collection: {collection_name}")